<a href="https://colab.research.google.com/github/rgaratac/dust_tool_test/blob/feature/simplified-tool/Dust_deduction_tool_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CAMS natural dust deduction tool

Removes the natural (desert dust) contribution from $PM_{10}$ and $PM_{2.5}$ measurements, as allowed by Article 16 of the Ambient Air Quality Directive (EU) 2024/2881. It combines EEA station measurements with CAMS dust forecasts and shows which stations meet the limit values with and without the deduction.

**How to use**

1. Run the **Setup** cell.
2. Fill in the **Settings** (section 1).
3. **First time:** keep `first_run = True` and run all cells. The EEA and CAMS data are downloaded (this can take a long time) and the results are saved.
4. **Next times:** set `first_run = False`. Nothing is downloaded: the saved results are loaded and you can go straight to the pairing (section 3) and the results (sections 4 and 5).

Files are saved in `CAMS_Tool_output` on Google Drive (Colab) or in the project folder (local).

Created by Jessie Zhang, TNO.

In [ ]:
#@title Setup (run once)
import importlib, importlib.util, subprocess, sys, urllib.request
from pathlib import Path
from importlib.metadata import version as _pkg_version
import requests

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if importlib.util.find_spec('cartopy') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'cartopy'])
if int(_pkg_version('plotly').split('.')[0]) < 6:  # Plotly 6 widgets also render in VS Code
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'plotly>=6', 'anywidget'])

GITHUB_BRANCH = 'feature/simplified-tool'
MODULES = ['util', 'cams_data_to_eea_daily', 'pairing', 'natural_contribution', 'pairing_viz',
           'dust_threshold_sensitivity', 'exceedance_report', 'dust_tool', 'dust_plots']
try:  # pin to the latest commit: raw.githubusercontent.com caches a branch for minutes
    _sha = requests.get(f'https://api.github.com/repos/rgaratac/dust_tool_test/commits/{GITHUB_BRANCH}',
                        headers={'Accept': 'application/vnd.github.sha'}, timeout=10).text.strip()
except Exception:
    _sha = ''
raw_base = f"https://raw.githubusercontent.com/rgaratac/dust_tool_test/{_sha if len(_sha) == 40 else GITHUB_BRANCH}"
for name in MODULES:
    if IN_COLAB or not Path(f'{name}.py').exists():  # local copies are kept
        urllib.request.urlretrieve(f'{raw_base}/{name}.py', f'{name}.py')
for name in MODULES:
    importlib.reload(importlib.import_module(name))

import matplotlib.pyplot as plt
from IPython.display import display
import dust_tool as dt
import dust_plots as dp
import pairing_viz as pv
print('Setup done.')

## 1. Settings

`countries`: two-letter codes separated by commas (e.g. `ES, PT`), or `ALL` for every country.

In [ ]:
#@title Main settings
first_run     = True    #@param {type:"boolean"}
year          = 2025    #@param {type:"integer"}
pollutant     = 'PM10'  #@param ["PM10", "PM2.5"]
dataset       = 'E1a'   #@param ["E1a", "E2a"]
temporal_flag = 'day'   #@param ["day", "hour"]
countries     = 'ES'    #@param {type:"string"}

settings = dt.Settings(
    first_run=first_run, year=year, pollutant=pollutant, dataset=dataset, temporal_flag=temporal_flag,
    countries=dt.ALL_COUNTRIES if countries.strip().upper() == 'ALL'
              else [c.strip().upper() for c in countries.split(',') if c.strip()],
)

| Setting | Meaning |
|---|---|
| `first_run` | `True`: download and compute everything. `False`: reuse the saved data and results. |
| `dataset` | `E1a`: verified data (previous years). `E2a`: up-to-date, unverified data (current year). |
| `temporal_flag` | `day`: EEA daily means, CAMS averaged over each station's local day. `hour`: EEA hourly data, daily means on UTC days. |

The defaults below follow the guidance; change them only if needed.

In [ ]:
#@title Advanced options (optional)
settings.cams_dust_threshold = 5      # µg/m³ CAMS dust above which a day is a dust day
settings.station_coverage    = 65     # % of the days of the year a station needs data

# Limit values, µg/m³ (Directive (EU) 2024/2881)
settings.pm10_daily_limit, settings.pm10_annual_limit = 45, 20
settings.pm25_daily_limit, settings.pm25_annual_limit = 25, 10

# Station pairing: 'default' pairs each station with a nearby reference, 'none' uses each station on its own
settings.pairing_mode           = 'default'
settings.max_rb_distance_km     = 100
settings.max_rb_altitude_diff_m = 500
settings.max_ub_distance_km     = 50
settings.use_user_pairing       = False   # True: start from your edited station_pairing_user.csv

# Your own observations instead of the EEA download (CSV files in the output folder)
settings.own_observations  = False
settings.observations_file = 'user_observations.csv'
settings.stations_file     = 'user_stations.csv'
settings.upload_own_files  = False        # Colab: choose the files on your computer

# Run switches: None = automatic from first_run; True/False forces a step
settings.download_eea       = None
settings.download_cams      = None
settings.compute_cams_daily = None
settings.load_parquet       = None

In [ ]:
tool = dt.DustTool(settings)   # prints what this run will do

## 2. Data

Downloads the EEA measurements and the CAMS dust data, computes the daily means and matches CAMS dust to each station.

* On the first CAMS download the tool asks once for your Copernicus ADS API key ([how to get it](https://cds.climate.copernicus.eu/how-to-api)).
* Station coordinates come from `DataExtract.csv`. When it is not in the output folder, the copy in the repository is downloaded.
* When saved results are found (`first_run = False`), this cell does nothing.

In [ ]:
tool.prepare_data()

## 3. Station pairing

Each station is classified as **RB** (rural background), **UB** (urban background) or **TH** (traffic/industrial hotspot). The natural dust load is measured at a reference station:

* **RB available** within the distance and altitude limits → its load is used (Option 1).
* **UB without RB** → the station itself is used (Option 2, self-calibration).
* **TH without RB** → the load of a nearby UB is used. Without a reference, nothing is subtracted.

In [ ]:
tool.build_pairing()

Click a station on the map to change its reference. Changes are saved to `station_pairing_user.csv` in the output folder.

In [ ]:
CLICKABLE_MAP = True   # False: static map, if the clickable one does not show
display(tool.pairing_app(clickable_map=CLICKABLE_MAP))

In [ ]:
#@title Edit the pairing as a CSV file (optional)
EXPORT_PAIRING_CSV = False   # write the pairing to station_pairing_user.csv (downloaded in Colab)
LOAD_PAIRING_CSV   = False   # read the edited station_pairing_user.csv back
UPLOAD_PAIRING_CSV = False   # Colab: choose the edited file on your computer

if EXPORT_PAIRING_CSV:
    tool.export_pairing_csv()
if LOAD_PAIRING_CSV:
    tool.load_pairing_csv(upload=UPLOAD_PAIRING_CSV)

## 4. Dust deduction

On every CAMS dust day the natural contribution is the measurement at the reference station minus its median on the surrounding non-dust days (±15 days for Option 1, ±3 days around the episode for Option 2). It is subtracted from the measurement, never below 0.

The results are saved (first run) or loaded (later runs), and summarised per station.

In [ ]:
station_table = tool.compute_results()

## 5. Results

### 5.1 Annual mean after the deduction
Colour: annual mean after the deduction. Size: days above the daily limit value that remain.

In [ ]:
dp.show_corrected_annual_map(tool)

### 5.2 Time series per station
Click a station on the map to see its time series. If the map does not work, use the next cell.

In [ ]:
dp.show_clickable_timeseries_map(tool, vmin=0, vmax=25)

In [ ]:
STATION = 0   # station id (e.g. 'ES/SP_28079004_10_6') or its position in the station table
fig, ax = dp.show_station_timeseries(tool, STATION)

### 5.3 Exceedance and annual maps
Observed values, the part removed by the deduction and limit value compliance: **green** meets without the deduction, **orange** meets only with it, **red** exceeds even with it.

In [ ]:
ALLOWED_DAILY_EXCEEDANCES = 18   # days per year allowed above the daily limit value
dp.show_exceedance_maps(tool, allowed_exceedances=ALLOWED_DAILY_EXCEEDANCES)
dp.show_annual_maps(tool)

### 5.4 Compliance per station
Table of every station with the annual mean and exceedance days, measured and after the deduction, and the status for both limit values. The table is saved as CSV in the output folder.

In [ ]:
SAVE_CSV = True
report_stations = dp.show_compliance_report(tool, ALLOWED_DAILY_EXCEEDANCES, save_csv=SAVE_CSV)
dp.show_compliance_maps(tool, report_stations, ALLOWED_DAILY_EXCEEDANCES)

## 6. Extras (optional)

### 6.1 Dust threshold sensitivity
Repeats the deduction for several CAMS dust thresholds and suggests the lowest one at which most CAMS dust days are confirmed by the measurements.

In [ ]:
RUN_THRESHOLD_SWEEP = False
THRESHOLDS = [2, 3, 4, 5, 6, 7, 8, 10]   # µg/m³

if RUN_THRESHOLD_SWEEP:
    consistency, suggestion = tool.threshold_sweep(THRESHOLDS)
    dp.show_threshold_sweep(tool, consistency, suggestion)

### 6.2 Hourly and daily results together
When both the `hour` and the `day` results are saved, this loads them together. Run the cells of section 5 again to plot both.

In [ ]:
COMBINE_HOURLY_DAILY = False
if COMBINE_HOURLY_DAILY:
    tool.combine_hourly_and_daily()